# Editing and orienting photos

Crop, flip, rotate, pad and thumbnail; convert channels and composite with alpha; and fix the EXIF orientation of phone photos. Every operation returns a new `Image`.

In [ ]:
import struct, zlib
import numpy as np
import matplotlib.pyplot as plt
import libstb

def show(images, titles=None, width=2.6):
    fig, axes = plt.subplots(1, len(images), figsize=(width * len(images), 2.6))
    for i, (ax, im) in enumerate(zip(np.atleast_1d(axes), images)):
        a = im.numpy() if hasattr(im, 'numpy') else im
        ax.imshow(a[..., 0] if a.shape[2] == 1 else a, cmap='gray' if a.shape[2] == 1 else None)
        ax.set_title(titles[i] if titles else '', fontsize=9); ax.axis('off')

In [ ]:
# A gradient with a white 'F' in the corner, so every flip and turn is visible.
y, x = np.mgrid[0:100, 0:160]
px = np.dstack([x * 255 // 159, y * 255 // 99, np.full_like(x, 80)]).astype(np.uint8)
px[8:48, 8:16] = 255; px[8:16, 8:36] = 255; px[24:32, 8:28] = 255
img = libstb.Image(px)
show([img], ["source"])

## Geometry

`rotate90` turns clockwise (a negative count turns the other way). `crop` needs a rectangle inside the image. `thumbnail` fits a box, keeps the aspect ratio and never enlarges.

In [ ]:
show(
    [img.crop(8, 8, 40, 40), img.flip_horizontal(), img.flip_vertical(), img.rotate90(), img.transpose(),
     img.pad(10, 10, 10, 10, fill=(255, 255, 255)), img.thumbnail(64, 64)],
    ["crop", "flip_horizontal", "flip_vertical", "rotate90", "transpose", "pad", "thumbnail 64"],
    width=2.2,
)

## Channels and alpha

`convert(n)` goes between 1 (gray), 2 (gray + alpha), 3 (RGB) and 4 (RGBA). `split` and `merge` take channels apart and put them back, in any order.

In [ ]:
r, g, b = img.split()
show([img.convert(1), r, g, b, libstb.Image.merge([b, g, r])],
     ["convert(1)", "R", "G", "B", "merge([B, G, R])"])

`composite(overlay, x, y)` blends with the "over" operator. The overlay may hang over the edge and may have any channel count. `flatten(background)` blends the alpha onto a colour and drops it.

In [ ]:
size = 90
yy, xx = np.mgrid[0:size, 0:size]
badge_px = np.zeros((size, size, 4), np.uint8)
badge_px[(xx - size / 2) ** 2 + (yy - size / 2) ** 2 < (size / 2 - 2) ** 2] = (220, 30, 30, 200)
badge = libstb.Image(badge_px)

show([img.composite(badge, 20, 15), img.composite(badge, 100, 40), badge.flatten(), badge.flatten((20, 40, 160))],
     ["composite at (20, 15)", "clipped at the edge", "flatten() on white", "flatten((20, 40, 160))"], width=3)

## EXIF orientation

A phone held sideways stores the picture turned and writes a tag (1 to 8) saying how to turn it back. stb ignores the tag; libstb applies it when asked, for JPEG and PNG. This cell writes the tag the way a phone does, so no camera is needed.

In [ ]:
def exif_tiff(orientation):
    header = b"II" + struct.pack("<HI", 42, 8)
    entry = struct.pack("<HHIHH", 0x0112, 3, 1, orientation, 0)
    return header + struct.pack("<H", 1) + entry + struct.pack("<I", 0)

def png_with_orientation(png, orientation):  # an eXIf chunk right after IHDR
    data = exif_tiff(orientation)
    chunk = struct.pack(">I", len(data)) + b"eXIf" + data + struct.pack(">I", zlib.crc32(b"eXIf" + data))
    return png[:33] + chunk + png[33:]

# Stored turned the opposite way and tagged 6 ("turn 90 degrees clockwise").
photo = png_with_orientation(img.rotate90(-1).to_png(), 6)

print("tag:", libstb.exif_orientation(photo))
raw = libstb.Image.open(photo)                 # the stored pixels (sideways)
upright = libstb.Image.open(photo, orient=True)  # the way it was meant to be seen
print("stored", raw.width, "x", raw.height, "| oriented", upright.width, "x", upright.height)
print("identical to the original:", np.array_equal(upright.numpy(), img.numpy()))
show([raw, upright], ["orient=False (default)", "orient=True"], width=3.5)

`orient(value)` applies any of the eight values to an `Image`; here is one picture seen with each:

In [ ]:
tiles = [img.pad(6, 6, 6, 6, fill=255).orient(o) for o in range(1, 9)]
show(tiles, [f"orient({o})" for o in range(1, 9)], width=2.0)